In [1]:
import pandas as pd
import zipfile
import os
from google.colab import files

uploaded = files.upload()

zip_files = [name for name in uploaded if name.lower().endswith(".zip")]

if not zip_files:
    raise ValueError("Please upload a ZIP file.")

zip_path = zip_files[0]
extract_folder = "/content/ecommerce_data"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

data = {}

for root, folders, file_names in os.walk(extract_folder):
    for file_name in file_names:
        if file_name.lower().endswith(".csv"):
            table_name = os.path.splitext(file_name)[0]
            file_path = os.path.join(root, file_name)
            data[table_name] = pd.read_csv(file_path)

if not data:
    raise ValueError("No CSV files were found inside the ZIP file.")

for table_name, df in data.items():
    print("\nTable:", table_name)
    print("Rows and columns:", df.shape)
    print("Column names:", list(df.columns))

Saving archive (1).zip to archive (1).zip

Table: OrderDetails
Rows and columns: (400, 6)
Column names: ['OrderDetailsID', 'ProductID', 'OrderItemQuantity', 'PerUnitPrice', 'OrderStatus', 'OrderID']

Table: Customer
Rows and columns: (400, 6)
Column names: ['CustomerID', 'CustomerName', 'CustomerEmail', 'CustomerPhone', 'CustomerAddress', 'CustomerCreditLimit']

Table: Region
Rows and columns: (400, 6)
Column names: ['RegionID', 'RegionName', 'CountryName', 'State', 'City', 'PostalCode']

Table: Employee
Rows and columns: (400, 7)
Column names: ['EmployeeID', 'EmployeeName', 'EmployeeEmail', 'EmployeePhone', 'EmployeeHireDate', 'EmployeeJobTitle', 'WarehouseID']

Table: Orders
Rows and columns: (400, 3)
Column names: ['OrderID', 'OrderDate', 'CustomerID']

Table: Product
Rows and columns: (400, 7)
Column names: ['ProductID', 'ProductName', 'CategoryName', 'ProductDescription', 'ProductStandardCost', 'ProductListPrice', 'Profit']

Table: Warehouse
Rows and columns: (400, 4)
Column names

In [2]:
for table_name, df in data.items():
    print("\nTable:", table_name)
    print(df.isnull().sum())


Table: OrderDetails
OrderDetailsID       0
ProductID            0
OrderItemQuantity    0
PerUnitPrice         0
OrderStatus          0
OrderID              0
dtype: int64

Table: Customer
CustomerID             0
CustomerName           0
CustomerEmail          0
CustomerPhone          0
CustomerAddress        0
CustomerCreditLimit    0
dtype: int64

Table: Region
RegionID       0
RegionName     0
CountryName    0
State          0
City           0
PostalCode     0
dtype: int64

Table: Employee
EmployeeID          0
EmployeeName        0
EmployeeEmail       0
EmployeePhone       0
EmployeeHireDate    0
EmployeeJobTitle    0
WarehouseID         0
dtype: int64

Table: Orders
OrderID       0
OrderDate     0
CustomerID    0
dtype: int64

Table: Product
ProductID              0
ProductName            0
CategoryName           0
ProductDescription     0
ProductStandardCost    0
ProductListPrice       0
Profit                 0
dtype: int64

Table: Warehouse
WarehouseID         0
WarehouseName 

In [3]:
for table_name, df in data.items():
    print("\nTable:", table_name)
    print("Duplicate rows:", df.duplicated().sum())


Table: OrderDetails
Duplicate rows: 0

Table: Customer
Duplicate rows: 0

Table: Region
Duplicate rows: 0

Table: Employee
Duplicate rows: 0

Table: Orders
Duplicate rows: 0

Table: Product
Duplicate rows: 0

Table: Warehouse
Duplicate rows: 0


In [4]:
id_columns = {
    "Employee": "EmployeeID",
    "Region": "RegionID",
    "Customer": "CustomerID",
    "Warehouse": "WarehouseID",
    "Product": "ProductID",
    "OrderDetails": "OrderDetailsID",
    "Orders": "OrderID"
}

for table_name, column_name in id_columns.items():
    df = data[table_name]
    print(
        table_name,
        "- Repeated IDs:",
        df[column_name].duplicated().sum()
    )

print(
    "\nOrderDetails - Repeated OrderIDs:",
    data["OrderDetails"]["OrderID"].duplicated().sum()
)

Employee - Repeated IDs: 0
Region - Repeated IDs: 0
Customer - Repeated IDs: 0
Warehouse - Repeated IDs: 0
Product - Repeated IDs: 0
OrderDetails - Repeated IDs: 0
Orders - Repeated IDs: 0

OrderDetails - Repeated OrderIDs: 0


In [5]:
orders = data["Orders"]
order_details = data["OrderDetails"]
products = data["Product"]

print("Negative quantities:", (order_details["OrderItemQuantity"] < 0).sum())
print("Zero quantities:", (order_details["OrderItemQuantity"] == 0).sum())

print("Zero or negative unit prices:", (order_details["PerUnitPrice"] <= 0).sum())

print("Negative standard costs:", (products["ProductStandardCost"] < 0).sum())
print("Negative list prices:", (products["ProductListPrice"] < 0).sum())

order_dates = pd.to_datetime(orders["OrderDate"], errors="coerce")
print("Invalid order dates:", order_dates.isna().sum())

print("Category names:")
print(sorted(products["CategoryName"].dropna().unique()))

Negative quantities: 0
Zero quantities: 0
Zero or negative unit prices: 0
Negative standard costs: 0
Negative list prices: 0
Invalid order dates: 0
Category names:
['CPU', 'Mother Board', 'RAM', 'Storage', 'Video Card']


In [6]:
print("Orders with unknown customers:",
      (~data["Orders"]["CustomerID"].isin(data["Customer"]["CustomerID"])).sum())

print("Order details with unknown orders:",
      (~data["OrderDetails"]["OrderID"].isin(data["Orders"]["OrderID"])).sum())

print("Order details with unknown products:",
      (~data["OrderDetails"]["ProductID"].isin(data["Product"]["ProductID"])).sum())

print("Employees with unknown warehouses:",
      (~data["Employee"]["WarehouseID"].isin(data["Warehouse"]["WarehouseID"])).sum())

print("Warehouses with unknown regions:",
      (~data["Warehouse"]["RegionID"].isin(data["Region"]["RegionID"])).sum())

Orders with unknown customers: 0
Order details with unknown orders: 0
Order details with unknown products: 0
Employees with unknown warehouses: 0
Warehouses with unknown regions: 0


In [7]:
import pandas as pd
import os

cleaned_folder = "/content/cleaned_data"
os.makedirs(cleaned_folder, exist_ok=True)

cleaned_data = {}

for table_name, df in data.items():
    cleaned_df = df.copy()

    for column in cleaned_df.select_dtypes(include="object").columns:
        cleaned_df[column] = cleaned_df[column].str.strip()

    cleaned_data[table_name] = cleaned_df

cleaned_data["Orders"]["OrderDate"] = pd.to_datetime(
    cleaned_data["Orders"]["OrderDate"],
    errors="coerce"
)

numeric_columns = {
    "OrderDetails": ["OrderItemQuantity", "PerUnitPrice"],
    "Product": ["ProductStandardCost", "ProductListPrice", "Profit"],
    "Customer": ["CustomerCreditLimit"]
}

for table_name, columns in numeric_columns.items():
    for column in columns:
        cleaned_data[table_name][column] = pd.to_numeric(
            cleaned_data[table_name][column],
            errors="coerce"
        )

for table_name, df in cleaned_data.items():
    df.to_csv(
        os.path.join(cleaned_folder, table_name + "_cleaned.csv"),
        index=False
    )

sales_data = cleaned_data["OrderDetails"].merge(
    cleaned_data["Orders"],
    on="OrderID",
    how="left"
)

sales_data = sales_data.merge(
    cleaned_data["Product"],
    on="ProductID",
    how="left"
)

sales_data = sales_data.merge(
    cleaned_data["Customer"],
    on="CustomerID",
    how="left"
)

sales_data.to_csv(
    os.path.join(cleaned_folder, "Sales_Data_Cleaned.csv"),
    index=False
)

print("Cleaned tables and combined sales dataset saved in:", cleaned_folder)
print("Combined sales dataset size:", sales_data.shape)

Cleaned tables and combined sales dataset saved in: /content/cleaned_data
Combined sales dataset size: (400, 19)


In [8]:
import pandas as pd
import os

summary = []

for table_name, df in data.items():
    cleaned_df = cleaned_data[table_name]

    summary.append({
        "Table": table_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Rows After Cleaning": len(cleaned_df)
    })

cleaning_summary = pd.DataFrame(summary)

print(cleaning_summary.to_string(index=False))

cleaning_summary.to_csv(
    "/content/cleaned_data/cleaning_summary.csv",
    index=False
)

print("\nCleaning summary saved successfully.")

       Table  Rows  Columns  Missing Values  Duplicate Rows  Rows After Cleaning
OrderDetails   400        6               0               0                  400
    Customer   400        6               0               0                  400
      Region   400        6               0               0                  400
    Employee   400        7               0               0                  400
      Orders   400        3               0               0                  400
     Product   400        7               0               0                  400
   Warehouse   400        4               0               0                  400

Cleaning summary saved successfully.


In [9]:
import pandas as pd
import os

orders = cleaned_data["Orders"]
details = cleaned_data["OrderDetails"]
products = cleaned_data["Product"]

checks = {
    "Negative quantities": int((details["OrderItemQuantity"] < 0).sum()),
    "Zero quantities": int((details["OrderItemQuantity"] == 0).sum()),
    "Zero or negative unit prices": int((details["PerUnitPrice"] <= 0).sum()),
    "Negative standard costs": int((products["ProductStandardCost"] < 0).sum()),
    "Negative list prices": int((products["ProductListPrice"] < 0).sum()),
    "Invalid order dates": int(orders["OrderDate"].isna().sum()),
    "Orders with unknown customers": int((~orders["CustomerID"].isin(cleaned_data["Customer"]["CustomerID"])).sum()),
    "Order details with unknown orders": int((~details["OrderID"].isin(orders["OrderID"])).sum()),
    "Order details with unknown products": int((~details["ProductID"].isin(products["ProductID"])).sum()),
    "Employees with unknown warehouses": int((~cleaned_data["Employee"]["WarehouseID"].isin(cleaned_data["Warehouse"]["WarehouseID"])).sum()),
    "Warehouses with unknown regions": int((~cleaned_data["Warehouse"]["RegionID"].isin(cleaned_data["Region"]["RegionID"])).sum())
}

validation_report = pd.DataFrame(
    list(checks.items()),
    columns=["Validation Check", "Number of Records"]
)

print(validation_report.to_string(index=False))

validation_report.to_csv(
    "/content/cleaned_data/validation_report.csv",
    index=False
)

print("\nValidation report saved successfully.")

                   Validation Check  Number of Records
                Negative quantities                  0
                    Zero quantities                  0
       Zero or negative unit prices                  0
            Negative standard costs                  0
               Negative list prices                  0
                Invalid order dates                  0
      Orders with unknown customers                  0
  Order details with unknown orders                  0
Order details with unknown products                  0
  Employees with unknown warehouses                  0
    Warehouses with unknown regions                  0

Validation report saved successfully.


In [10]:
cleaning_decisions = """
E-Commerce Dataset Cleaning Report

1. Dataset Overview
The dataset contains seven tables: Employee, Region, Customer, Warehouse, Product, OrderDetails, and Orders.

2. Missing Values
All seven tables were checked for missing values. No missing values were found.

3. Duplicate Records
All seven tables were checked for exact duplicate rows. No exact duplicate rows were found.

4. Text Cleaning
Leading and trailing spaces were removed from text columns in the cleaned copies.

5. Data Types
OrderDate was converted to a datetime format. Selected quantity, price, cost, profit, and credit limit columns were converted to numeric values.

6. Validation
Quantity, price, date, and table relationship checks were performed. The results are recorded in validation_report.csv. Any flagged records should be reviewed before deciding whether to correct or remove them.

7. Data Preservation
The original data was kept unchanged. Cleaned tables and the combined sales dataset were saved separately.

8. Output Files
Cleaned tables, Sales_Data_Cleaned.csv, cleaning_summary.csv, and validation_report.csv were saved in the cleaned_data folder.
"""

report_path = "/content/cleaned_data/cleaning_decisions.txt"

with open(report_path, "w") as file:
    file.write(cleaning_decisions.strip())

print("Cleaning decisions report saved to:", report_path)

Cleaning decisions report saved to: /content/cleaned_data/cleaning_decisions.txt
